# Qwen2.5-1.5B + LoRA 微调模型对话

直接运行下方单元格即可

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print('加载模型中...')
tok = AutoTokenizer.from_pretrained(
    '/Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct',
    trust_remote_code=True,
)
tok.pad_token = tok.eos_token

base = AutoModelForCausalLM.from_pretrained(
    '/Users/jinqianyu/Documents/个人资料/大数据/财经博主/models/Qwen2.5-1.5B-Instruct',
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map='mps',
)

model = PeftModel.from_pretrained(
    base,
    '/Users/jinqianyu/Documents/个人资料/大数据/财经博主/outputs/qwen_lora_finetuned',
)
model.eval()
print('✅ 加载完成！')

def ask(q):
    text = tok.apply_chat_template(
        [{'role':'user','content':q}],
        tokenize=False,
        add_generation_prompt=True,
    )
    out = model.generate(
        **tok(text, return_tensors='pt').to('mps'),
        max_new_tokens=200,
        do_sample=False,
    )
    return tok.decode(
        out[0][len(tok(text,return_tensors='pt')['input_ids'][0]):],
        skip_special_tokens=True,
    )

# 测试
for q in ['什么是LPR？对房贷有什么影响？','什么是市盈率（PE）？']:
    print(f'\n📌 问: {q}')
    print(f'💡 答: {ask(q)}')

/Users/jinqianyu/Documents/LLaMA-Factory/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


加载模型中...


Loading weights: 100%|█████████████████████████████████████████████████████████████████| 338/338 [00:03<00:00, 107.97it/s]


✅ 加载完成！

📌 问: 什么是LPR？对房贷有什么影响？
💡 答: LPR（贷款市场报价利率）是由多家银行自主报出的，由全国银行间同业拆借中心计算得出的加权平均利率。LPR是金融机构向客户发放贷款时参考的基准利率，主要分为1年期和5年期以上两个期限品种。LPR的变化直接影响到房贷利率：如果LPR下降，意味着商业银行的成本降低，可以将更低的LPR传导给购房者；反之，如果LPR上升，则意味着房贷成本增加。对于不同类型的房贷（如首套房、二套房），LPR变化的影响程度也有所不同。例如，首套房的LPR下调通常会直接降低月供压力，而二套房则需要综合考虑其他因素。

📌 问: 什么是市盈率（PE）？
💡 答: 市盈率（P/E Ratio，Price-to-Earnings Ratio），也称动态市盈率或滚动市盈率，是股价与每股收益的比率。计算公式为：市盈率 = 每股价格 / 每股收益。简单来说，就是用公司股票当前的价格除以每股盈利得到的结果。市盈率越高意味着投资者愿意支付更高的价格来购买每单位盈利，反映了市场对公司未来盈利能力的预期。通常认为，市盈率在20倍以下被认为是安全边际，超过30倍则可能被高估。需要注意的是，市盈率受行业和周期性因素影响较大，不同行业的平均市盈率差异很大。


In [ ]:
# 交互式对话
print('💬 输入 quit 退出')
while True:
    q = input('\n👤 ')
    if q.lower() in ('quit','exit','q'): break
    print('🤖', ask(q))

💬 输入 quit 退出
